# 🧠 Entrenamiento Modelo Trading Solana

**Objetivo:** Crear un modelo simple que prediga si el precio subirá/bajará en los próximos 5 minutos.

**Fase 2 del Sistema** - PC Escritorio

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import asyncpg
import asyncio

In [ ]:
import os
# Cargar datos desde la base de datos de la Pi
async def load_trades(days=7):
    conn = await asyncpg.connect(
        host='192.168.1.X',  # IP de tu Raspberry Pi
        port=5432,
        database='solana_trading',
        user='solana_user',
        password=os.environ['DB_PASSWORD']  # defínela antes: set DB_PASSWORD=...
    )
    
    query = f"""
    SELECT * FROM solana_trades 
    WHERE timestamp > NOW() - INTERVAL '{days} days'
    ORDER BY timestamp
    """
    
    df = await conn.fetchdf(query)
    await conn.close()
    return df

In [ ]:
# Cargar datos
print("📡 Descargando datos desde la Pi...")
trades = asyncio.run(load_trades(days=7))
print(f"✅ Descargados {len(trades)} trades")

In [ ]:
# Crear features técnicas
def create_features(df):
    df = df.copy()
    
    # RSI (simplificado)
    delta = df['price'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # Media móvil
    df['sma_20'] = df['price'].rolling(window=20).mean()
    df['sma_50'] = df['price'].rolling(window=50).mean()
    
    # Volumen
    df['volume_ma_20'] = df['size'].rolling(window=20).mean()
    
    # Target: ¿Subirá en los próximos 5 minutos?
    df['future_price'] = df['price'].shift(-60)  # Aproximadamente 5 min
    df['target'] = (df['future_price'] > df['price']).astype(int)
    
    return df.dropna()

In [ ]:
# Aplicar features
print("🔧 Creando features...")
df_features = create_features(trades)
print(f"✅ Datos procesados: {len(df_features)} filas")

In [ ]:
# Preparar datos para entrenamiento
feature_cols = ['rsi', 'sma_20', 'volume_ma_20']
X = df_features[feature_cols]
y = df_features['target']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
# Entrenar modelo Random Forest
print("🎯 Entrenando modelo...")
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Evaluar
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n📊 Accuracy: {accuracy:.2%}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Exportar a ONNX
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)

with open('../models/solana_model_v1.onnx', 'wb') as f:
    from onnx import save
    save(onnx_model, f)

print("✅ Modelo exportado a models/solana_model_v1.onnx")

In [ ]:
# Enviar a la Pi (opcional)
!scp ../models/solana_model_v1.onnx pi@192.168.1.X:/home/pi/trading_system/models/